In [0]:
-- Databricks Catalog Setup Script
-- Creates catalog, schema, and three tables with proper relationships

-- Create catalog (if not exists)
CREATE CATALOG IF NOT EXISTS ecommerce_catalog;

-- Use the catalog
USE CATALOG ecommerce_catalog;

-- Create schema
CREATE SCHEMA IF NOT EXISTS bookstore
COMMENT "Schema for bookstore business data";

ALTER TABLE orders SET TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported')

-- Use the schema
USE SCHEMA bookstore;

-- ============================================
-- Create customers table
-- ============================================
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    email STRING NOT NULL,
    profile STRING,
    updated TIMESTAMP, --DEFAULT CURRENT_TIMESTAMP,
    created_date TIMESTAMP --DEFAULT CURRENT_TIMESTAMP
)
COMMENT "Customer information for bookstore";

-- ============================================
-- Create books table
-- ============================================
DROP TABLE IF EXISTS books;

CREATE TABLE books (
    book_id BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    title STRING NOT NULL,
    author STRING NOT NULL,
    category STRING,
    price DECIMAL(10, 2) NOT NULL,
    created_date TIMESTAMP --DEFAULT CURRENT_TIMESTAMP
)
COMMENT "Books catalog with pricing and details";

-- ============================================
-- Create orders table with books as JSON array
-- ============================================
DROP TABLE IF EXISTS orders;

CREATE TABLE orders (
    order_id BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    timestamp TIMESTAMP NOT NULL,
    customer_id BIGINT NOT NULL,
    quantity INT NOT NULL,
    total DECIMAL(10, 2) NOT NULL,
    books ARRAY<STRUCT<
        bookId INT,
        quantity INT,
        subtotal DECIMAL(10, 2)
    >> NOT NULL,
    created_date TIMESTAMP,-- DEFAULT CURRENT_TIMESTAMP,
    CONSTRAINT fk_customer_id FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
COMMENT "Customer orders with nested JSON format for books";

-- Create indexes for better query performance
CREATE INDEX idx_orders_customer ON orders(customer_id);
CREATE INDEX idx_orders_timestamp ON orders(timestamp);
CREATE INDEX idx_customers_email ON customers(email);
